# 01 — The position network

Exploration of the aggregate transition network across all `final` matches: PageRank hubs,
Markov reward-risk, communities, and fighter similarity. All logic lives in
`analysis.network_metrics` / `analysis.fighter_similarity`; this notebook just visualises.
Run from the repo root (`uv run jupyter lab`); needs a `.env` with the DB connection.

In [ ]:
from dotenv import load_dotenv; load_dotenv()
import pandas as pd, matplotlib.pyplot as plt, networkx as nx
from db.base import db_session
from analysis.network_metrics import (build_transition_network, node_centralities,
    reward_risk_ranking, reward_risk_with_ci, detect_communities,
    route_to_submission, pagerank_ranking, weighted_pagerank_ranking)
from analysis.fighter_similarity import athlete_style_vectors, nearest_in

with db_session() as s:
    G = build_transition_network(s)
    ids, mat = athlete_style_vectors(s)
print(G.number_of_nodes(), 'positions,', G.number_of_edges(), 'transition types')

## Hubs of grappling — PageRank

In [ ]:
pd.DataFrame(pagerank_ranking(G, 15), columns=['position', 'pagerank'])

In [ ]:
print('Weighted PageRank (Zhang et al. 2022) — edge-weight aware:')
w = pd.DataFrame(weighted_pagerank_ranking(G, 15), columns=['position', 'weighted_pagerank'])
display(w)
print('\nDelta (weighted − standard):')
p = pd.DataFrame(pagerank_ranking(G, 999), columns=['position', 'pagerank'])
merged = p.merge(w, on='position', how='outer').fillna(0)
merged['delta'] = merged.weighted_pagerank - merged.pagerank
display(merged.sort_values('delta', ascending=False).head(10))

## Reward-risk balance (Lamas et al. 2024)

In [ ]:
# Standard reward-risk ranking (Lamas et al. 2024)
rr = reward_risk_ranking(G, min_occ=5, limit=15)
for n, _, _ in rr[:6]:
    print(n, ' → ', ' → '.join(route_to_submission(G, n)[1:]))
display(pd.DataFrame(rr, columns=['position', 'reward_risk', 'seen']))

# Bayesian reward-risk with 95% credible interval (same data)
rr_ci = reward_risk_with_ci(G, min_occ=5, limit=15)
display(pd.DataFrame(rr_ci, columns=['position', 'point_est', 'ci_lower', 'ci_upper', 'occ']))

## Game families — community detection

In [ ]:
for i, c in enumerate(detect_communities(G, min_occ=3)[:6], 1):
    print(f'Family {i} ({len(c)}):', ', '.join(c[:10]))

## The network — top-30 positions, sized by PageRank vs weighted PageRank

In [ ]:
# Standard PageRank sizing
cent = node_centralities(G)
top = [n for n, _ in pagerank_ranking(G, 30)]
H = G.subgraph(top)
pos = nx.spring_layout(H, seed=1, k=0.6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))

# Left: standard PageRank
sizes1 = [cent[n]['pagerank'] * 18000 for n in H.nodes]
ax1.set_title('Top 30 — Standard PageRank', fontsize=13)
nx.draw_networkx_edges(H, pos, alpha=0.15, arrows=False, ax=ax1)
nx.draw_networkx_nodes(H, pos, node_size=sizes1, node_color='#4d86ff', alpha=0.8, ax=ax1)
nx.draw_networkx_labels(H, pos, font_size=7, ax=ax1)
ax1.axis('off')

# Right: weighted PageRank (Zhang et al. 2022)
wpr = dict(weighted_pagerank_ranking(G, 999))
sizes2 = [wpr.get(n, 0) * 18000 for n in H.nodes]
ax2.set_title('Top 30 — Weighted PageRank', fontsize=13)
nx.draw_networkx_edges(H, pos, alpha=0.15, arrows=False, ax=ax2)
nx.draw_networkx_nodes(H, pos, node_size=sizes2, node_color='#ff6b4d', alpha=0.8, ax=ax2)
nx.draw_networkx_labels(H, pos, font_size=7, ax=ax2)
ax2.axis('off')

plt.tight_layout(); plt.show()

## Fighter DNA — stylistically closest athletes

In [ ]:
for _, name in ids[:8]:
    print(name, '→', nearest_in(ids, mat, name, 3))